# 07. Combining the three fold models

**What this notebook does:** takes the saved test logits of the three CV fold models from notebook 06 and combines them in two basic ways, then writes one Kaggle CSV for each:

- **average:** the fold models' log-probabilities and CRF transition scores are averaged, and the averaged scores are decoded once (Viterbi).
- **vote:** each fold model is decoded on its own, and a location is kept when at least 2 of the 3 models predict it.

**How well does the combination work?** It cannot be measured fairly on the CV folds. Each held-out fold was seen in training by the other two fold models, so scoring a combination there would leak. The only unseen text common to all three models is the 26-post labeling sample, which is tiny and noisy. The real check is the leaderboard, so this notebook only prepares the files and shows how much the folds agree.

Nothing is trained here. It reads `results/06_final_cv_training.jsonl` and the saved `test.npz` and `labeling.npz` files, so it takes seconds.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/ai301")
    ROOT = PROJECT / "deep-learning-assignment3"
    BRANCH = "model"  # the branch that holds model/; main does not have it
    if not ROOT.exists():
        PROJECT.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "-b", BRANCH, "https://github.com/koalla05/deep-learning-assignment3.git", str(ROOT)], check=True)
    else:
        subprocess.run(["git", "fetch", "origin", BRANCH], cwd=ROOT, check=True)
        subprocess.run(["git", "checkout", BRANCH], cwd=ROOT, check=True)
        subprocess.run(["git", "pull", "origin", BRANCH], cwd=ROOT, check=True)
    # The frozen split manifests are stored with Git LFS; a plain clone only has small pointer files.
    subprocess.run("apt-get install -y -q git-lfs", shell=True)
    subprocess.run(["git", "lfs", "install"], cwd=ROOT)
    subprocess.run(["git", "lfs", "pull"], cwd=ROOT)
    import urllib.request

    for name in ("role_manifest_v2.parquet", "cv_manifest_v1.parquet"):
        manifest = ROOT / "eda" / "output" / name
        if manifest.read_bytes()[:4] != b"PAR1":  # still a pointer file: download the real one
            url = f"https://media.githubusercontent.com/media/koalla05/deep-learning-assignment3/{BRANCH}/eda/output/{name}"
            urllib.request.urlretrieve(url, manifest)
        assert manifest.read_bytes()[:4] == b"PAR1", f"{name} is not a parquet file; download it from GitHub and upload it to {manifest.parent}"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "lightning", "pytorch-crf", "tokenizers"], check=True)
    os.environ["NER_DATA_DIR"] = str(PROJECT / "data" / "kse-ua-location-extraction-2026")
    os.environ["NER_WORK_DIR"] = str(PROJECT / "ner_work")
else:
    ROOT = Path.cwd()
    while not (ROOT / "eda").exists():
        ROOT = ROOT.parent
    # Results downloaded from Colab (folder ner_work) are continued here: finished folds are skipped.
    if "NER_WORK_DIR" not in os.environ and (ROOT / "data" / "ner_work").exists():
        os.environ["NER_WORK_DIR"] = str(ROOT / "data" / "ner_work")

import torch

os.environ.setdefault("NER_WANDB", "off")
os.environ["NER_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"
sys.path[:0] = [str(ROOT), str(ROOT / "model")]
print("Device:", os.environ["NER_DEVICE"], "| project:", ROOT, "| work folder:", os.environ.get("NER_WORK_DIR", ROOT / "model"))

Device: cpu | project: /Users/allakovalcuk/AI301/deep-learning-assignment3 | work folder: /Users/allakovalcuk/AI301/deep-learning-assignment3/data/ner_work


In [2]:
import json
from collections import Counter

import pandas as pd
from IPython.display import display

from nerkit.artifacts import combine_models, decode_split, load_logits, majority_vote, row_key, write_submission
from nerkit.config import WORK_DIR
from nerkit.data import load_labeling_sample, load_test
from nerkit.decisions import DECODING
from nerkit.experiment import read_results
from nerkit.metrics import evaluate

SOURCE = "06_final_cv_training"
DECODE = {name: DECODING[name] for name in ("merge", "drop_empty", "trim_number", "bias")}
records = sorted(read_results(SOURCE, raw=True), key=lambda record: record["config"]["fold"])
assert len(records) == 3, "run notebook 06 first: it must have finished all three folds"
print("Fold models:", {record["name"]: record["key"] for record in records})
print("Work folder:", WORK_DIR)

Fold models: {'fold0': '367d29e13b0b', 'fold1': '60d6b400897e', 'fold2': 'c048151b9c64'}
Work folder: /Users/allakovalcuk/AI301/deep-learning-assignment3/data/ner_work


## Test predictions of each fold and of the two combinations

In [3]:
test_rows = load_test()
test_texts = {row_key(row): row["text"] for row in test_rows}
loaded = [load_logits(record["key"], "test") for record in records]
models, metas = [item[0] for item in loaded], [item[1] for item in loaded]

fold_predictions = [decode_split(model, test_rows, crf=meta["crf"], **DECODE) for model, meta in zip(models, metas)]
average_logits, average_crf = combine_models(models, metas, test_texts)
average_predictions = decode_split(average_logits, test_rows, crf=average_crf, **DECODE)
vote_predictions = majority_vote(fold_predictions, 2)

named = {f"fold {record['config']['fold']}": predictions for record, predictions in zip(records, fold_predictions)}
named["average of 3"] = average_predictions
named["vote, 2 of 3"] = vote_predictions
display(pd.DataFrame({name: dict(posts_with_locations=sum(bool(p) for p in predictions),
                                 locations=sum(len(p) for p in predictions))
                      for name, predictions in named.items()}).T)

,posts_with_locations,locations
fold 0,317,701
fold 1,333,749
fold 2,326,758
average of 3,322,734
"vote, 2 of 3",323,712


## How much do the fold models agree?

For every location string predicted by at least one fold model, how many of the three models predicted it in that post. Strings that all three agree on are the confident ones; a vote of 2 of 3 drops the ones only a single model predicts.

In [4]:
agreement = Counter()
for lists in zip(*fold_predictions):
    for count in Counter(value for values in lists for value in set(values)).values():
        agreement[count] += 1
total = sum(agreement.values())
display(pd.DataFrame({"locations": [agreement[n] for n in (3, 2, 1)],
                      "share": [agreement[n] / total for n in (3, 2, 1)]},
                     index=["predicted by all 3", "by exactly 2", "by only 1"]).round(3))

,locations,share
predicted by all 3,598,0.666
by exactly 2,114,0.127
by only 1,186,0.207


## Rough check on the 26-post labeling sample

Competition posts that none of the models saw. With only 26 posts this is too noisy to choose between methods, but it shows whether a combination is broadly in line with the single models.

In [5]:
labeling_rows = load_labeling_sample()
labeling_texts = {row_key(row): row["text"] for row in labeling_rows}
labeling = [load_logits(record["key"], "labeling") for record in records]
label_models, label_metas = [item[0] for item in labeling], [item[1] for item in labeling]
label_folds = [decode_split(model, labeling_rows, crf=meta["crf"], **DECODE) for model, meta in zip(label_models, label_metas)]
label_average, label_crf = combine_models(label_models, label_metas, labeling_texts)
label_named = {f"fold {record['config']['fold']}": predictions for record, predictions in zip(records, label_folds)}
label_named["average of 3"] = decode_split(label_average, labeling_rows, crf=label_crf, **DECODE)
label_named["vote, 2 of 3"] = majority_vote(label_folds, 2)

rows = []
for name, predictions in label_named.items():
    score = evaluate(labeling_rows, predictions)["pooled"]
    rows.append(dict(method=name, precision=score["precision"], recall=score["recall"], f1=score["f1"]))
display(pd.DataFrame(rows).set_index("method").round(3))

,precision,recall,f1
method,,,
fold 0,0.588,0.625,0.606
fold 1,0.650,0.812,0.722
fold 2,0.595,0.781,0.676
average of 3,0.625,0.781,0.694
"vote, 2 of 3",0.641,0.781,0.704


## Write the submission files

Both files are checked against `test.csv` (same ids and order, every prediction an exact substring of its text). Upload them on the competition page and note the public scores below.

In [11]:
for name, predictions in (("ensemble_average", average_predictions), ("ensemble_vote2", vote_predictions)):
    path = write_submission(WORK_DIR / "submissions" / f"{name}.csv", test_rows, predictions)
    print(f"{path.name}: {sum(bool(p) for p in predictions)} of {len(predictions)} posts have locations, "
          f"{sum(len(p) for p in predictions)} locations in total")

LEADERBOARD = {"fold 0": 0.57098, "fold 1": 0.58308, "fold 2": 0.58594,  # public scores from Kaggle
               "average of 3": 0.58787, "vote, 2 of 3": 0.58914}
(WORK_DIR / "final_cv" / "ensemble_summary.json").write_text(json.dumps(dict(
    decode=DECODE, folds=[record["key"] for record in records], leaderboard=LEADERBOARD,
    labeling_f1=[row for row in rows]), indent=1), encoding="utf-8")

ensemble_average.csv: 322 of 477 posts have locations, 734 locations in total
ensemble_vote2.csv: 323 of 477 posts have locations, 712 locations in total


900

## Leaderboard results

Public scores of all five uploaded files. The combinations are only slightly above the best single fold: the average gains about +0.002 over the best fold (+0.008 over the fold mean), and the 2-of-3 vote gains about +0.003 over the best fold. These differences are far smaller than the gap between the local CV score (0.89) and the leaderboard (0.58), so combining folds is not what limits the score; the mismatch between the training sentences and the competition posts is.

In [12]:
board = pd.Series(LEADERBOARD, name="public_f1")
display(board.to_frame().round(5))
singles = board[[name for name in board.index if name.startswith("fold")]]
print(f"single folds: mean {singles.mean():.4f}, best {singles.max():.4f}")
for name in ("average of 3", "vote, 2 of 3"):
    print(f"{name}: {board[name]:.4f} ({board[name] - singles.mean():+.4f} vs fold mean, {board[name] - singles.max():+.4f} vs best fold)")

,public_f1
fold 0,0.57098
fold 1,0.58308
fold 2,0.58594
average of 3,0.58787
"vote, 2 of 3",0.58914


single folds: mean 0.5800, best 0.5859
average of 3: 0.5879 (+0.0079 vs fold mean, +0.0019 vs best fold)
vote, 2 of 3: 0.5891 (+0.0091 vs fold mean, +0.0032 vs best fold)
